In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
print("=" * 50)
print("  Case Study 1: MNIST Digit Recognition (MLP)")
print("=" * 50)

  Case Study 1: MNIST Digit Recognition (MLP)


In [3]:
# ── 1. Load & Preprocess ──────────────────────────────
print("\n[1/5] Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data / 255.0
y = mnist.target.astype(int).reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y_enc = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42
)
print(f"    Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")


[1/5] Loading MNIST dataset...
    Train: 56000 samples | Test: 14000 samples


In [4]:
# ── 2. Activation Functions ───────────────────────────
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy_loss(y_pred, y_true):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))


In [5]:
# ── 3. Weight Initialisation (He Init) ───────────────
print("\n[2/5] Initialising network weights...")
np.random.seed(42)
input_size, h1, h2, output_size = 784, 128, 64, 10

W1 = np.random.randn(input_size, h1) * np.sqrt(2.0 / input_size)
b1 = np.zeros((1, h1))
W2 = np.random.randn(h1, h2)         * np.sqrt(2.0 / h1)
b2 = np.zeros((1, h2))
W3 = np.random.randn(h2, output_size) * np.sqrt(2.0 / h2)
b3 = np.zeros((1, output_size))

print("    Architecture: 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax)")



[2/5] Initialising network weights...
    Architecture: 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax)


In [6]:
# ── 4. Training Loop (Mini-batch GD) ─────────────────
print("\n[3/5] Training for 20 epochs...")
learning_rate = 0.1   # ← increased; safe now that gradients are normalized
epochs = 20
batch_size = 256
n_samples = X_train.shape[0]

for epoch in range(epochs):
    idx = np.random.permutation(n_samples)
    X_shuf, y_shuf = X_train[idx], y_train[idx]
    epoch_loss = 0
    n_batches  = 0

    for start in range(0, n_samples, batch_size):
        Xb = X_shuf[start:start + batch_size]
        yb = y_shuf[start:start + batch_size]
        m  = Xb.shape[0]          # ← actual batch size (last batch may differ)

        # ── Forward pass ──
        Z1 = Xb @ W1 + b1;  A1 = relu(Z1)
        Z2 = A1  @ W2 + b2;  A2 = relu(Z2)
        Z3 = A2  @ W3 + b3;  A3 = softmax(Z3)

        epoch_loss += cross_entropy_loss(A3, yb)
        n_batches  += 1

        # ── Backpropagation (divide by m to get mean gradient) ──
        dZ3 = (A3 - yb) / m                        # ✅ FIX: normalize by batch size
        dW3 = A2.T @ dZ3
        db3 = dZ3.sum(axis=0, keepdims=True)

        dA2 = dZ3 @ W3.T
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = A1.T @ dZ2
        db2 = dZ2.sum(axis=0, keepdims=True)

        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = Xb.T @ dZ1
        db1 = dZ1.sum(axis=0, keepdims=True)

        # ── Weight update ──
        W3 -= learning_rate * dW3;  b3 -= learning_rate * db3
        W2 -= learning_rate * dW2;  b2 -= learning_rate * db2
        W1 -= learning_rate * dW1;  b1 -= learning_rate * db1

    # ── Epoch accuracy on training set ──
    Z1 = X_train @ W1 + b1;  A1 = relu(Z1)
    Z2 = A1 @ W2 + b2;        A2 = relu(Z2)
    Z3 = A2 @ W3 + b3;        A3 = softmax(Z3)
    train_acc = np.mean(np.argmax(A3, axis=1) == np.argmax(y_train, axis=1))
    print(f"    Epoch {epoch+1:2d}/20 | Loss: {epoch_loss/n_batches:.4f} | Train Acc: {train_acc*100:.2f}%")


[3/5] Training for 20 epochs...
    Epoch  1/20 | Loss: 0.5633 | Train Acc: 90.44%
    Epoch  2/20 | Loss: 0.2718 | Train Acc: 93.27%
    Epoch  3/20 | Loss: 0.2157 | Train Acc: 94.42%
    Epoch  4/20 | Loss: 0.1812 | Train Acc: 95.16%
    Epoch  5/20 | Loss: 0.1567 | Train Acc: 95.93%
    Epoch  6/20 | Loss: 0.1379 | Train Acc: 96.44%
    Epoch  7/20 | Loss: 0.1235 | Train Acc: 96.85%
    Epoch  8/20 | Loss: 0.1116 | Train Acc: 97.17%
    Epoch  9/20 | Loss: 0.1016 | Train Acc: 97.51%
    Epoch 10/20 | Loss: 0.0924 | Train Acc: 97.40%
    Epoch 11/20 | Loss: 0.0859 | Train Acc: 97.47%
    Epoch 12/20 | Loss: 0.0788 | Train Acc: 97.96%
    Epoch 13/20 | Loss: 0.0738 | Train Acc: 97.63%
    Epoch 14/20 | Loss: 0.0680 | Train Acc: 98.20%
    Epoch 15/20 | Loss: 0.0641 | Train Acc: 98.48%
    Epoch 16/20 | Loss: 0.0596 | Train Acc: 98.58%
    Epoch 17/20 | Loss: 0.0562 | Train Acc: 98.68%
    Epoch 18/20 | Loss: 0.0523 | Train Acc: 98.76%
    Epoch 19/20 | Loss: 0.0489 | Train Acc: 98.70

In [8]:
# ── 5. Test Evaluation ────────────────────────────────
print("\n[4/5] Evaluating on test set...")
Z1 = X_test @ W1 + b1;  A1 = relu(Z1)
Z2 = A1 @ W2 + b2;       A2 = relu(Z2)
Z3 = A2 @ W3 + b3;       A3 = softmax(Z3)

predictions = np.argmax(A3, axis=1)
true_labels  = np.argmax(y_test, axis=1)
test_accuracy = np.mean(predictions == true_labels)

print(f"\n{'='*50}")
print(f"  ✅  Test Accuracy: {test_accuracy*100:.2f}%")
print(f"{'='*50}")

# ── 6. Per-digit Accuracy ─────────────────────────────
print("\n[5/5] Per-digit accuracy:")
for digit in range(10):
    mask = true_labels == digit
    acc  = np.mean(predictions[mask] == digit)
    bar  = "█" * int(acc * 20)
    print(f"    Digit {digit}: {acc*100:5.1f}%  {bar}")


[4/5] Evaluating on test set...

  ✅  Test Accuracy: 97.39%

[5/5] Per-digit accuracy:
    Digit 0:  98.4%  ███████████████████
    Digit 1:  98.8%  ███████████████████
    Digit 2:  97.2%  ███████████████████
    Digit 3:  96.7%  ███████████████████
    Digit 4:  97.3%  ███████████████████
    Digit 5:  96.7%  ███████████████████
    Digit 6:  98.7%  ███████████████████
    Digit 7:  97.8%  ███████████████████
    Digit 8:  95.9%  ███████████████████
    Digit 9:  96.1%  ███████████████████


In [ ]:
# Let us chnge the architecture to this and check
# 784 → 256 → 128 → 10

In [9]:
# ── 3. Xavier Initialisation ─────────────────────────
# Xavier: W ~ Uniform(-sqrt(6/(fan_in+fan_out)), +sqrt(6/(fan_in+fan_out)))
# Best suited for symmetric activations; works well with ReLU too
print("\n[2/5] Initialising weights (Xavier)...")
np.random.seed(42)
input_size, h1, h2, output_size = 784, 128, 64, 10

def xavier_init(fan_in, fan_out):
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, (fan_in, fan_out))

W1 = xavier_init(input_size, h1)
b1 = np.zeros((1, h1))
W2 = xavier_init(h1, h2)
b2 = np.zeros((1, h2))
W3 = xavier_init(h2, output_size)
b3 = np.zeros((1, output_size))

print("    Architecture : 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax)")
print("    Init         : Xavier Uniform")
print("    Optimiser    : Adam (lr=0.001, β1=0.9, β2=0.999)")


[2/5] Initialising weights (Xavier)...
    Architecture : 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax)
    Init         : Xavier Uniform
    Optimiser    : Adam (lr=0.001, β1=0.9, β2=0.999)


In [10]:
# ── 4. Adam Optimiser State ───────────────────────────
# Adam tracks two moving averages per parameter:
#   m = 1st moment (mean of gradients)       — β1 decay
#   v = 2nd moment (mean of squared grads)   — β2 decay
# Bias-corrected estimates prevent near-zero updates early in training

beta1, beta2, epsilon = 0.9, 0.999, 1e-8
lr = 0.001
t  = 0   # global timestep (increments every batch)

# Initialise moment vectors (same shape as each weight/bias)
m_W1, v_W1 = np.zeros_like(W1), np.zeros_like(W1)
m_b1, v_b1 = np.zeros_like(b1), np.zeros_like(b1)
m_W2, v_W2 = np.zeros_like(W2), np.zeros_like(W2)
m_b2, v_b2 = np.zeros_like(b2), np.zeros_like(b2)
m_W3, v_W3 = np.zeros_like(W3), np.zeros_like(W3)
m_b3, v_b3 = np.zeros_like(b3), np.zeros_like(b3)

def adam_update(param, grad, m, v, t):
    """One Adam step. Returns updated param, m, v."""
    m = beta1 * m + (1 - beta1) * grad          # update biased 1st moment
    v = beta2 * v + (1 - beta2) * (grad ** 2)   # update biased 2nd moment
    m_hat = m / (1 - beta1 ** t)                 # bias correction
    v_hat = v / (1 - beta2 ** t)                 # bias correction
    param -= lr * m_hat / (np.sqrt(v_hat) + epsilon)
    return param, m, v


In [11]:
# ── 5. Training Loop ──────────────────────────────────
print("\n[3/5] Training for 20 epochs...")
epochs     = 20
batch_size = 256
n_samples  = X_train.shape[0]

for epoch in range(epochs):
    idx = np.random.permutation(n_samples)
    X_shuf, y_shuf = X_train[idx], y_train[idx]
    epoch_loss = 0
    n_batches  = 0

    for start in range(0, n_samples, batch_size):
        Xb = X_shuf[start:start + batch_size]
        yb = y_shuf[start:start + batch_size]
        m_batch = Xb.shape[0]
        t += 1   # increment global timestep

        # ── Forward pass ──
        Z1 = Xb @ W1 + b1;  A1 = relu(Z1)
        Z2 = A1  @ W2 + b2;  A2 = relu(Z2)
        Z3 = A2  @ W3 + b3;  A3 = softmax(Z3)

        epoch_loss += cross_entropy_loss(A3, yb)
        n_batches  += 1

        # ── Backpropagation ──
        dZ3 = (A3 - yb) / m_batch
        dW3 = A2.T @ dZ3
        db3 = dZ3.sum(axis=0, keepdims=True)

        dA2 = dZ3 @ W3.T
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = A1.T @ dZ2
        db2 = dZ2.sum(axis=0, keepdims=True)

        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = Xb.T @ dZ1
        db1 = dZ1.sum(axis=0, keepdims=True)

        # ── Adam weight updates ──
        W3, m_W3, v_W3 = adam_update(W3, dW3, m_W3, v_W3, t)
        b3, m_b3, v_b3 = adam_update(b3, db3, m_b3, v_b3, t)
        W2, m_W2, v_W2 = adam_update(W2, dW2, m_W2, v_W2, t)
        b2, m_b2, v_b2 = adam_update(b2, db2, m_b2, v_b2, t)
        W1, m_W1, v_W1 = adam_update(W1, dW1, m_W1, v_W1, t)
        b1, m_b1, v_b1 = adam_update(b1, db1, m_b1, v_b1, t)

    # ── Epoch accuracy ──
    Z1 = X_train @ W1 + b1;  A1 = relu(Z1)
    Z2 = A1 @ W2 + b2;        A2 = relu(Z2)
    Z3 = A2 @ W3 + b3;        A3 = softmax(Z3)
    train_acc = np.mean(np.argmax(A3, axis=1) == np.argmax(y_train, axis=1))
    print(f"    Epoch {epoch+1:2d}/20 | Loss: {epoch_loss/n_batches:.4f} | Train Acc: {train_acc*100:.2f}%")



[3/5] Training for 20 epochs...
    Epoch  1/20 | Loss: 0.4502 | Train Acc: 94.20%
    Epoch  2/20 | Loss: 0.1754 | Train Acc: 96.28%
    Epoch  3/20 | Loss: 0.1221 | Train Acc: 96.79%
    Epoch  4/20 | Loss: 0.0958 | Train Acc: 97.89%
    Epoch  5/20 | Loss: 0.0756 | Train Acc: 98.23%
    Epoch  6/20 | Loss: 0.0613 | Train Acc: 98.75%
    Epoch  7/20 | Loss: 0.0521 | Train Acc: 98.79%
    Epoch  8/20 | Loss: 0.0437 | Train Acc: 99.16%
    Epoch  9/20 | Loss: 0.0367 | Train Acc: 99.33%
    Epoch 10/20 | Loss: 0.0299 | Train Acc: 99.17%
    Epoch 11/20 | Loss: 0.0259 | Train Acc: 99.52%
    Epoch 12/20 | Loss: 0.0225 | Train Acc: 99.63%
    Epoch 13/20 | Loss: 0.0189 | Train Acc: 99.65%
    Epoch 14/20 | Loss: 0.0158 | Train Acc: 99.66%
    Epoch 15/20 | Loss: 0.0139 | Train Acc: 99.66%
    Epoch 16/20 | Loss: 0.0107 | Train Acc: 99.89%
    Epoch 17/20 | Loss: 0.0085 | Train Acc: 99.86%
    Epoch 18/20 | Loss: 0.0076 | Train Acc: 99.84%
    Epoch 19/20 | Loss: 0.0077 | Train Acc: 99.83

In [12]:
# ── 6. Test Evaluation ────────────────────────────────
print("\n[4/5] Evaluating on test set...")
Z1 = X_test @ W1 + b1;  A1 = relu(Z1)
Z2 = A1 @ W2 + b2;       A2 = relu(Z2)
Z3 = A2 @ W3 + b3;       A3 = softmax(Z3)

predictions   = np.argmax(A3, axis=1)
true_labels   = np.argmax(y_test, axis=1)
test_accuracy = np.mean(predictions == true_labels)

print(f"\n{'='*50}")
print(f"  ✅  Test Accuracy: {test_accuracy*100:.2f}%")
print(f"{'='*50}")

# ── 7. Per-digit Accuracy ─────────────────────────────
print("\n[5/5] Per-digit accuracy:")
for digit in range(10):
    mask = true_labels == digit
    acc  = np.mean(predictions[mask] == digit)
    bar  = "█" * int(acc * 20)
    print(f"    Digit {digit}: {acc*100:5.1f}%  {bar}")



[4/5] Evaluating on test set...

  ✅  Test Accuracy: 97.41%

[5/5] Per-digit accuracy:
    Digit 0:  97.8%  ███████████████████
    Digit 1:  98.0%  ███████████████████
    Digit 2:  97.6%  ███████████████████
    Digit 3:  97.6%  ███████████████████
    Digit 4:  96.8%  ███████████████████
    Digit 5:  97.1%  ███████████████████
    Digit 6:  99.4%  ███████████████████
    Digit 7:  98.1%  ███████████████████
    Digit 8:  95.1%  ███████████████████
    Digit 9:  96.5%  ███████████████████
